## Bibliotecas

In [1]:
# Biblioteca para trabalhar com arquivos, pastas e caminhos do sistema operacional
import os
import csv
from datetime import datetime

## Arquivos

In [2]:
pasta = "1-lh_nautical_csv"
arquivos = os.listdir(pasta) # Lista o conteúdo da pasta especificada

arquivos

['addresses.csv',
 'attributes.csv',
 'brands.csv',
 'categories.csv',
 'customers.csv',
 'employees.csv',
 'fiscal_invoices.csv',
 'goods_receipts.csv',
 'goods_receipt_items.csv',
 'locations.csv',
 'orders.csv',
 'order_items.csv',
 'payments.csv',
 'products.csv',
 'product_suppliers.csv',
 'product_variants.csv',
 'purchase_orders.csv',
 'purchase_order_items.csv',
 'returns.csv',
 'return_items.csv',
 'stock_levels.csv',
 'stock_movements.csv',
 'suppliers.csv',
 'variant_attribute_values.csv']

In [3]:
arquivos_csv = [
    arquivo
    for arquivo in arquivos
    if arquivo.endswith(".csv") # Lista apenas os arquivos que terminam com .csv, caso haja outros tipos de arquivos na pasta
]

print(f"Número de arquivos CSV: {len(arquivos_csv)}")
arquivos_csv

Número de arquivos CSV: 24


['addresses.csv',
 'attributes.csv',
 'brands.csv',
 'categories.csv',
 'customers.csv',
 'employees.csv',
 'fiscal_invoices.csv',
 'goods_receipts.csv',
 'goods_receipt_items.csv',
 'locations.csv',
 'orders.csv',
 'order_items.csv',
 'payments.csv',
 'products.csv',
 'product_suppliers.csv',
 'product_variants.csv',
 'purchase_orders.csv',
 'purchase_order_items.csv',
 'returns.csv',
 'return_items.csv',
 'stock_levels.csv',
 'stock_movements.csv',
 'suppliers.csv',
 'variant_attribute_values.csv']

## Valores das colunas

In [4]:
# Receba as linhas, o cabeçalho e o nome de uma coluna e devolve todos os valores dessa coluna
def obter_valores_coluna(linhas, cabecalho, nome_coluna):

    indice = cabecalho.index(nome_coluna)

    valores = []

    for linha in linhas:
        valores.append(linha[indice])

    return valores

## Tipo de dado

In [5]:
def detectar_tipo(valor):

    if valor.isdigit():
        numero = int(valor)

        if numero > 2147483647:
            return "BIGINT"

        return "INTEGER"

    try:
        datetime.strptime(valor, "%Y-%m-%d %H:%M:%S")
        return "TIMESTAMP"
    except:
        pass

    try:
        datetime.strptime(valor, "%Y-%m-%d")
        return "DATE"
    except:
        pass

    try:
        float(valor)
        return "NUMERIC"
    except:
        return "TEXT"

In [6]:
print(detectar_tipo("123"))
print(detectar_tipo("123.45"))
print(detectar_tipo("2025-01-01"))
print(detectar_tipo("2025-01-01 10:30:00"))
print(detectar_tipo("online"))
print(detectar_tipo("ORD-000001"))

INTEGER
NUMERIC
DATE
TIMESTAMP
TEXT
TEXT


## Tipo de dado por coluna

In [7]:
# Identifica tipo de coluna com base nos valores contidos nela
def detectar_tipo_coluna(valores):

    tipos = []

    for valor in valores:

        if valor.strip() == "": # Se o valor for uma string vazia (apenas espaços em branco), ele será ignorado e o loop continuará para o próximo valor
            continue

        tipos.append(detectar_tipo(valor))

    tipos_unicos = set(tipos)

    if tipos_unicos == {"INTEGER"}:
        return "INTEGER"

    if tipos_unicos <= {"INTEGER", "NUMERIC"}:
        return "NUMERIC"

    if tipos_unicos == {"DATE"}:
        return "DATE"

    if tipos_unicos == {"TIMESTAMP"}:
        return "TIMESTAMP"

    return "TEXT"

## Create table

In [8]:
def gerar_create_table(nome_tabela, cabecalho, linhas):
    schema = {}

    for coluna in cabecalho:
        valores = obter_valores_coluna(linhas, cabecalho, coluna)
        tipo = detectar_tipo_coluna(valores)
        schema[coluna] = tipo

    colunas_sql = []
    for coluna, tipo in schema.items():
        colunas_sql.append(f'"{coluna}" {tipo}')

    colunas_sql_texto = ",\n".join(colunas_sql)

    sql = f'''CREATE TABLE IF NOT EXISTS "{nome_tabela}" (
{colunas_sql_texto}
);'''

    return sql

## Inteiração dos arquivos

In [9]:
todos_os_sql = []

for arquivo_csv in arquivos_csv:
    caminho = os.path.join(pasta, arquivo_csv)

    with open(caminho, "r", encoding="utf-8-sig", newline="") as arquivo:
        leitor = csv.reader(arquivo)
        cabecalho = next(leitor)
        linhas = list(leitor)

    nome_tabela = os.path.splitext(arquivo_csv)[0] # Remove a extensão do arquivo para usar como nome da tabela

    sql_tabela = gerar_create_table(
        nome_tabela,
        cabecalho,
        linhas
    )

    todos_os_sql.append(sql_tabela)

In [10]:
print( f"Número de arquivos CSV: {len(todos_os_sql)}")
todos_os_sql

Número de arquivos CSV: 24


['CREATE TABLE IF NOT EXISTS "addresses" (\n"id" INTEGER,\n"customer_id" INTEGER,\n"address_type" TEXT,\n"postal_code" TEXT,\n"street" TEXT,\n"number" INTEGER,\n"complement" TEXT,\n"district" TEXT,\n"city" TEXT,\n"state" TEXT,\n"country" TEXT,\n"is_primary" TEXT\n);',
 'CREATE TABLE IF NOT EXISTS "attributes" (\n"id" INTEGER,\n"name" TEXT,\n"data_type" TEXT\n);',
 'CREATE TABLE IF NOT EXISTS "brands" (\n"id" INTEGER,\n"name" TEXT,\n"country" TEXT,\n"is_active" TEXT,\n"created_at" TIMESTAMP,\n"updated_at" TIMESTAMP\n);',
 'CREATE TABLE IF NOT EXISTS "categories" (\n"id" INTEGER,\n"name" TEXT,\n"slug" TEXT,\n"parent_category_id" INTEGER,\n"is_active" TEXT,\n"created_at" TIMESTAMP,\n"updated_at" TIMESTAMP\n);',
 'CREATE TABLE IF NOT EXISTS "customers" (\n"id" INTEGER,\n"person_type" TEXT,\n"legal_name" TEXT,\n"trade_name" TEXT,\n"tax_id" TEXT,\n"state_registration" TEXT,\n"email" TEXT,\n"phone" TEXT,\n"is_active" TEXT,\n"created_at" TIMESTAMP,\n"updated_at" TIMESTAMP\n);',
 'CREATE TABLE 

In [11]:
schema_sql = "\n\n".join(todos_os_sql)

## Schema

In [12]:
with open("schema_final.sql", "w", encoding="utf-8") as arquivo:
    arquivo.write(schema_sql)

In [13]:
print(schema_sql.count("CREATE TABLE"))

24


In [14]:
print(schema_sql[:1000])

CREATE TABLE IF NOT EXISTS "addresses" (
"id" INTEGER,
"customer_id" INTEGER,
"address_type" TEXT,
"postal_code" TEXT,
"street" TEXT,
"number" INTEGER,
"complement" TEXT,
"district" TEXT,
"city" TEXT,
"state" TEXT,
"country" TEXT,
"is_primary" TEXT
);

CREATE TABLE IF NOT EXISTS "attributes" (
"id" INTEGER,
"name" TEXT,
"data_type" TEXT
);

CREATE TABLE IF NOT EXISTS "brands" (
"id" INTEGER,
"name" TEXT,
"country" TEXT,
"is_active" TEXT,
"created_at" TIMESTAMP,
"updated_at" TIMESTAMP
);

CREATE TABLE IF NOT EXISTS "categories" (
"id" INTEGER,
"name" TEXT,
"slug" TEXT,
"parent_category_id" INTEGER,
"is_active" TEXT,
"created_at" TIMESTAMP,
"updated_at" TIMESTAMP
);

CREATE TABLE IF NOT EXISTS "customers" (
"id" INTEGER,
"person_type" TEXT,
"legal_name" TEXT,
"trade_name" TEXT,
"tax_id" TEXT,
"state_registration" TEXT,
"email" TEXT,
"phone" TEXT,
"is_active" TEXT,
"created_at" TIMESTAMP,
"updated_at" TIMESTAMP
);

CREATE TABLE IF NOT EXISTS "employees" (
"id" INTEGER,
"full_name" TEXT,
"